<a href="https://colab.research.google.com/github/valanchick/Enterprise-Grade-Movie-Recommender-System/blob/main/Rec_sys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommender System (CatBoostRanker + PySpark + FastAPI)

Этот проект демонстрирует полный цикл создания рекомендательной системы:
1. **ELT (Extract, Load, Transform)** с использованием `PySpark` для генерации фичей пользователей и фильмов.
2. **Data Preparation**: Генерация негативных примеров (Negative Sampling) с помощью `pandas`.
3. **Model Training**: Обучение ML-модели ранжирования `CatBoostRanker` (YetiRank).
4. **Inference**: Класс для получения рекомендаций.
5. **API**: Обертка инференса в микросервис на `FastAPI`.

In [ ]:
!pip install -q pyspark catboost fastapi uvicorn nest-asyncio

import os
import urllib.request
import zipfile

os.makedirs('./data', exist_ok=True)
os.makedirs('./models', exist_ok=True)

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "./data/ml-latest-small.zip"

if not os.path.exists("./data/ratings.csv"):
    print("Скачивание датасета MovieLens...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("./data/")

    os.rename("./data/ml-latest-small/ratings.csv", "./data/ratings.csv")
    os.rename("./data/ml-latest-small/movies.csv", "./data/movies.csv")
    print("Датасет загружен и готов к работе!")
else:
    print("Датасет уже существует.")

Датасет уже существует.


In [ ]:
from pyspark.sql import SparkSession

def run_pyspark_elt():
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("RecSys_ELT") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

    df_ratings = spark.read.csv("./data/ratings.csv", header=True, inferSchema=True)
    df_movies = spark.read.csv("./data/movies.csv", header=True, inferSchema=True)

    df_ratings.createOrReplaceTempView("ratings")
    df_movies.createOrReplaceTempView("movies")

    user_features = spark.sql("""
        select
            userId,
            count(*) as user_total_ratings,
            avg(rating) as user_mean_rating,
            stddev(rating) as user_std_rating
        from ratings
        group by userId
    """)
    user_features.createOrReplaceTempView("user_features")

    item_features = spark.sql("""
        select
            movieId,
            count(userId) as item_total_views,
            avg(rating) as item_mean_rating
        from ratings
        group by movieId
    """)
    item_features.createOrReplaceTempView("movie_features")

    favorite_genre_df = spark.sql("""
        WITH exploded_movies AS (
            SELECT movieId, genre FROM movies LATERAL VIEW explode(split(genres, '[|]')) t AS genre
        ),
        user_genre_counts AS (
            SELECT r.userId, em.genre, COUNT(*) as genre_count
            FROM ratings r
            JOIN exploded_movies em ON r.movieId = em.movieId
            WHERE r.rating >= 3.5
            GROUP BY r.userId, em.genre
        ),
        ranked_genres AS (
            SELECT userId, genre, ROW_NUMBER() OVER (PARTITION BY userId ORDER BY genre_count DESC) AS rn
            FROM user_genre_counts
        )
        SELECT userId, genre AS user_favorite_genre
        FROM ranked_genres WHERE rn = 1
    """)
    favorite_genre_df.createOrReplaceTempView("user_favorite_genres")

    final_df = spark.sql("""
        SELECT
            r.userId,
            r.movieId,
            r.rating,
            uf.user_total_ratings,
            uf.user_mean_rating,
            COALESCE(uf.user_std_rating, 0) as user_std_rating,
            COALESCE(ufg.user_favorite_genre, 'Unknown') as user_favorite_genre,
            iff.item_total_views,
            iff.item_mean_rating,
            m.genres as movie_genres_str,
            CASE WHEN array_contains(split(m.genres, '[|]'), ufg.user_favorite_genre) THEN 1 ELSE 0 END as is_genre_match
        FROM ratings r
        LEFT JOIN movies m ON r.movieId = m.movieId
        LEFT JOIN user_features uf ON r.userId = uf.userId
        LEFT JOIN movie_features iff ON r.movieId = iff.movieId
        LEFT JOIN user_favorite_genres ufg ON r.userId = ufg.userId
    """)

    final_df.write.mode("overwrite").parquet("./data/processed_data.parquet")
    print(f"ELT завершен. Размер данных: {final_df.count()} строк")
    spark.stop()

run_pyspark_elt()

ELT завершен. Размер данных: 100836 строк


In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

def generate_negatives(df, num_negatives=1):
    all_movie_ids = df['movieId'].unique()
    user_seen_movies = df.groupby('userId')['movieId'].agg(set).to_dict()

    negative_samples = []

    for user_id, seen_set in tqdm(user_seen_movies.items(), desc="Генерация негативных примеров"):
        count_needed = len(seen_set) * num_negatives

        for _ in range(count_needed):
            while True:
                candidate_id = np.random.choice(all_movie_ids)
                if candidate_id not in seen_set:
                    break

            negative_samples.append({
                'userId': user_id,
                'movieId': candidate_id,
                'rating': 0,
                'is_negative': 1
            })

    return pd.DataFrame(negative_samples)

def prepare_training_data():
    df_positive = pd.read_parquet("./data/processed_data.parquet")
    df_positive['is_negative'] = 0

    df_negative = generate_negatives(df_positive, num_negatives=1)
    df_full = pd.concat([df_positive, df_negative], ignore_index=True)

    user_cols = ['userId', 'user_total_ratings', 'user_mean_rating', 'user_std_rating', 'user_favorite_genre']
    item_cols = ['movieId', 'item_total_views', 'item_mean_rating', 'movie_genres_str']

    user_lookup = df_positive[user_cols].drop_duplicates('userId').set_index('userId')
    item_lookup = df_positive[item_cols].drop_duplicates('movieId').set_index('movieId')

    for col in user_cols:
        if col != 'userId':
            df_full[col] = df_full['userId'].map(user_lookup[col])

    for col in item_cols:
        if col != 'movieId':
            df_full[col] = df_full['movieId'].map(item_lookup[col])

    df_full['is_genre_match'] = df_full.apply(
        lambda x: 1 if str(x['user_favorite_genre']) in str(x['movie_genres_str']) else 0, axis=1
    )

    df_full = df_full.fillna(0)
    df_full.to_parquet("./data/train_set.parquet")
    print(f"Подготовка данных завершена. Итоговый размер: {df_full.shape}")

prepare_training_data()

Генерация негативных примеров:   0%|          | 0/610 [00:00<?, ?it/s]

Подготовка данных завершена. Итоговый размер: (201672, 12)


In [ ]:
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns

def train_model():
    df = pd.read_parquet("./data/train_set.parquet")
    df['user_favorite_genre'] = df['user_favorite_genre'].astype(str)

    cat_features = ['user_favorite_genre']
    num_features = [
        'user_total_ratings', 'user_mean_rating', 'user_std_rating',
        'item_total_views', 'item_mean_rating', 'is_genre_match'
    ]

    feature_names = cat_features + num_features
    df = df.sort_values(by='userId')

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(df, groups=df['userId']))

    train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

    train_pool = Pool(
        data=train_df[feature_names], label=train_df['rating'],
        group_id=train_df['userId'], cat_features=cat_features
    )
    test_pool = Pool(
        data=test_df[feature_names], label=test_df['rating'],
        group_id=test_df['userId'], cat_features=cat_features
    )

    model = CatBoostRanker(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function='YetiRank',
        eval_metric='NDCG:top=10',
        random_seed=42,
        verbose=50
    )

    model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)
    model.save_model("./models/recsys_model.cbm")

    importances = model.get_feature_importance(train_pool)
    feature_imp = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(by='importance', ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(x='importance', y='feature', data=feature_imp, palette="viridis")
    plt.title("Важность признаков (Feature Importance)")
    plt.show()

train_model()

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.6638876	best: 0.6638876 (0)	total: 4.57s	remaining: 22m 45s
50:	test: 0.8473666	best: 0.8473666 (50)	total: 3m 1s	remaining: 14m 45s
100:	test: 0.8469999	best: 0.8492554 (83)	total: 5m 58s	remaining: 11m 46s


In [ ]:
class Recommender:
    def __init__(self, model_path, data_path, movies_csv_path):
        self.model = CatBoostRanker()
        self.model.load_model(model_path)

        self.features = pd.read_parquet(data_path)
        self.movies_df = pd.read_csv(movies_csv_path)

        self.user_features = self.features[['userId', 'user_total_ratings', 'user_mean_rating', 'user_std_rating', 'user_favorite_genre']].drop_duplicates('userId').set_index('userId')
        self.item_features = self.features[['movieId', 'item_total_views', 'item_mean_rating', 'movie_genres_str']].drop_duplicates('movieId').set_index('movieId')

        self.all_movie_ids = self.item_features.index.values

    def predict(self, user_id, top_k=5):
        if user_id not in self.user_features.index:
            return []

        u_feats = self.user_features.loc[user_id]
        candidates = pd.DataFrame({'movieId': self.all_movie_ids})
        candidates['userId'] = user_id

        for col in u_feats.index:
            candidates[col] = u_feats[col]

        candidates = candidates.join(self.item_features, on='movieId')
        candidates['is_genre_match'] = candidates.apply(
            lambda x: 1 if str(x['user_favorite_genre']) in str(x['movie_genres_str']) else 0, axis=1
        )

        feature_cols = [
            'user_favorite_genre', 'user_total_ratings', 'user_mean_rating',
            'user_std_rating', 'item_total_views', 'item_mean_rating', 'is_genre_match'
        ]

        X = candidates[feature_cols]
        candidates['score'] = self.model.predict(X)

        top_movies = candidates.sort_values(by='score', ascending=False).head(top_k)

        result = pd.merge(top_movies, self.movies_df, on='movieId')[['movieId', 'title', 'score']]
        return result

rec_service = Recommender(
    model_path="./models/recsys_model.cbm",
    data_path="./data/train_set.parquet",
    movies_csv_path="./data/movies.csv"
)

print("Рекомендации для пользователя ID 1")
display(rec_service.predict(user_id=1, top_k=5))

Рекомендации для пользователя ID 1


,movieId,title,score
0,2959,Fight Club (1999),3.208480
1,260,Star Wars: Episode IV - A New Hope (1977),3.133401
2,2571,"Matrix, The (1999)",3.083129
3,1198,Raiders of the Lost Ark (Indiana Jones and the...,3.057275
4,2028,Saving Private Ryan (1998),3.008281


In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio
from threading import Thread
import requests
import time

app = FastAPI(title="Movie RecSys API")

@app.get("/")
def health_check():
    return {"status": "ok", "message": "RecSys Service is running"}

@app.get("/recommend/{user_id}")
def get_recommendations(user_id: int):
    try:
        result_df = rec_service.predict(user_id, top_k=5)
        if len(result_df) == 0:
            return {"error": "User not found"}

        recommendations = result_df[['movieId', 'title']].to_dict(orient="records")
        return {
            "user_id": user_id,
            "recommendations": recommendations
        }
    except Exception as e:
        return {"error": str(e)}

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = Thread(target=run_server)
server_thread.start()

time.sleep(3)

print("Ответ от health check:")
print(requests.get("http://127.0.0.1:8000/").json())

print("\nОтвет от эндпоинта рекомендаций (User 42):")
import json
response = requests.get("http://127.0.0.1:8000/recommend/42").json()
print(json.dumps(response, indent=2, ensure_ascii=False))

Ответ от health check:
{'status': 'ok', 'message': 'RecSys Service is running'}

Ответ от эндпоинта рекомендаций (User 42):
{
  "user_id": 42,
  "recommendations": [
    {
      "movieId": 318,
      "title": "Shawshank Redemption, The (1994)"
    },
    {
      "movieId": 2959,
      "title": "Fight Club (1999)"
    },
    {
      "movieId": 356,
      "title": "Forrest Gump (1994)"
    },
    {
      "movieId": 858,
      "title": "Godfather, The (1972)"
    },
    {
      "movieId": 527,
      "title": "Schindler's List (1993)"
    }
  ]
}
